# Extra · scaling: datos vs parámetros

> **No es el eje del módulo.** Sirve para oír la frase del ebook: *no preentrenas GPT-4; compras el pan*.

## Dummy

Dos perillas que bajan la pérdida, hasta que el cómputo o el overfit paran la fiesta:

1. **Más cerebro** (más capas, más $d$ → más parámetros $N$).
2. **Más libro** (fracción del Quijote → más tokens $D$).

Kaplan (2020) y Chinchilla (Hoffmann et al., 2022) midieron esto en un rango enorme: la pérdida cae como una **potencia** de $N$ y de $D$. Tú no vas a estimar el exponente con 200 KB en una laptop. Sí vas a ver la *tendencia*: a veces un modelo chico con más datos gana a un modelo grande con hambre.

La moraleja del diplomado no es “entonces preentrena”. Es: esa curva ya la pagó otro. Nosotros bajamos Qwen 0.5B (un punto de la curva) y afinamos.

## Matemática (forma de Kaplan, ilustrativa)

$$
\mathcal{L}(N,D)\;\approx\;E_\infty+\Bigl(\frac{N_c}{N}\Bigr)^{\alpha}+\Bigl(\frac{D_c}{D}\Bigr)^{\beta}.
$$

$E_\infty$ es el piso: el lenguaje es inherentemente impredecible; la pérdida **nunca** llega a 0. $N_c,D_c,\alpha,\beta$ son constantes empíricas. Chinchilla añade: para un presupuesto de FLOPs, hay un $N$ y un $D$ *pareja* — modelo gordo con poco texto desperdicia cómputo.

En este notebook: tres tamaños `(small, medium, large)` × tres fracciones de corpus `(0.25, 0.50, 1.00)`. Misma receta, distinta carne.

## Cómo leer el resumen

Para cada `data_frac`, ¿cuál modelo gana val_loss? Si `small` + 100% datos pega a `large` + 25% datos, esa es la lección de Chinchilla en miniatura. No extrapoles a GPT-4; extrapola a “no recortes el JSONL de Don Titular para ahorrarte 10 minutos”.


In [ ]:
# Colab: descomenta las dos líneas de abajo (una vez).
# %pip install -q torch
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/datos/quijote.txt

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from pathlib import Path

# Local: ../datos/quijote.txt (o ../../ desde extra/). Colab: el wget lo deja junto al notebook.
TEXT_PATH = next(
    (str(p) for p in (Path("quijote.txt"), Path("../datos/quijote.txt"), Path("../../datos/quijote.txt")) if p.exists()),
    "quijote.txt",
)
print(f"device={DEVICE}  archivo={TEXT_PATH}")

import time
from typing import Dict, Any


In [ ]:
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().replace("\r\n", "\n").replace("\r", "\n")


def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


In [ ]:
class CharWindowDataset(Dataset):
    """Ventanas x → y corrido +1. Si max_windows no es None, muestrea inicios al azar.

    Dummy: el Quijote tiene ~2M letras. Recorrer TODAS las ventanas en CPU no cabe
    en una sesión. Muestrear M inicios es el mismo examen, con menos filas.
    """

    def __init__(self, data_ids: torch.Tensor, seq_len: int, max_windows: Optional[int] = 8000, seed: int = 0):
        self.data = data_ids
        self.seq_len = seq_len
        n = max(0, len(self.data) - self.seq_len - 1)
        if max_windows is not None and n > max_windows:
            g = torch.Generator().manual_seed(seed)
            self.index = torch.randint(0, n, (max_windows,), generator=g)
        else:
            self.index = torch.arange(n)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i: int):
        idx = int(self.index[i])
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int = 128, n_head: int = 4, n_layer: int = 2, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.seq_len, "T excede seq_len del modelo"
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos.unsqueeze(0))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(1, n)


@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total += loss.item()
        n += 1
    return total / max(1, n)


@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, seq_len: int, max_new_tokens: int, temperature: float, device: str):
    model.eval()
    device = torch.device(device)
    ids = [stoi[ch] for ch in prompt if ch in stoi] or [0]
    out = ids[:]
    for _ in range(max_new_tokens):
        ctx = out[-seq_len:]
        x = torch.tensor(ctx, dtype=torch.long, device=device).unsqueeze(0)
        logits = model(x)[0, -1, :] / max(1e-8, temperature)
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1).item()
        out.append(next_id)
    return decode(out, itos)


In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def subtext_prefix(ids: torch.Tensor, frac: float) -> torch.Tensor:
    m = max(4000, int(len(ids) * frac))
    return ids[: min(m, len(ids))]


@dataclass
class ScalingConfig:
    seq_len: int = 64
    n_head: int = 4
    dropout: float = 0.1
    batch_size: int = 32
    lr: float = 3e-4
    epochs: int = 2
    max_windows: int = 2500
    device: str = DEVICE
    data_fracs: Tuple[float, ...] = (0.25, 0.50, 1.00)
    models: Tuple[Tuple[str, int, int], ...] = (
        ("small", 2, 128),
        ("medium", 3, 192),
        ("large", 4, 256),
    )


def run_scaling(text_path: str, cfg: ScalingConfig):
    # vocab del texto ENTERO, para que los tres tamaños hablen el mismo abecedario
    text = load_text(text_path)
    stoi, _ = build_vocab(text)
    ids_full = encode(text, stoi)
    device = torch.device(cfg.device)
    vocab = len(stoi)
    results = []

    for frac in cfg.data_fracs:
        ids = subtext_prefix(ids_full, frac)
        split = int(0.9 * len(ids))
        train_ds = CharWindowDataset(ids[:split], cfg.seq_len, cfg.max_windows, seed=0)
        val_ds = CharWindowDataset(ids[split:], cfg.seq_len, min(600, cfg.max_windows), seed=1)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

        for label, n_layer, n_embd in cfg.models:
            model = MiniTransformerLM(vocab, cfg.seq_len, n_embd, cfg.n_head, n_layer, cfg.dropout).to(device)
            params = count_params(model)
            optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
            t0 = time.time()
            stable = True
            try:
                for _ in range(cfg.epochs):
                    tr = train_one_epoch(model, train_loader, optim, device)
                    if not math.isfinite(tr):
                        stable = False
                        break
                vl = eval_loss(model, val_loader, device) if stable else float("inf")
            except Exception:
                stable = False
                vl = float("inf")
            dt = time.time() - t0
            ppl = math.exp(vl) if (stable and vl < 20) else float("inf")
            rec = {
                "data_frac": frac, "model": label, "layers": n_layer, "embd": n_embd,
                "params": params, "val_loss": vl, "val_ppl": ppl, "time_s": round(dt, 2),
                "stable": stable, "train_windows": len(train_ds),
            }
            results.append(rec)
            print(
                f"data={int(frac*100):3d}% | {label:6s} (L={n_layer},d={n_embd},P={params/1e6:.2f}M) "
                f"| stable={stable} | val_loss={vl:.4f} | ppl={ppl:.2f} | t={dt:.1f}s"
            )
    return sorted(results, key=lambda r: r["val_loss"]), results


cfg = ScalingConfig()
print("CPU: 2 épocas, ventanas recortadas. GPU: sube epochs y max_windows.")
best_sorted, all_results = run_scaling(TEXT_PATH, cfg)
print("\nTOP 10 (menor val_loss)")
for r in best_sorted[:10]:
    print(r)
print("\nMejor modelo por fracción de datos")
for frac in cfg.data_fracs:
    subset = sorted([r for r in all_results if r["data_frac"] == frac], key=lambda x: x["val_loss"])
    b = subset[0]
    print(f"data={int(frac*100):3d}% → {b['model']}  val_loss={b['val_loss']:.4f}  t={b['time_s']}s")
